# PySpark analysis flow using a reviews dataset

##  Import Libraries and Initialize Spark

In [3]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
# Initialize Spark Session
spark = SparkSession.builder \
    .appName("Amazon Review Analysis") \
    .getOrCreate()


## Load The Dataset

In [5]:
# Load the dataset
df = spark.read.csv("amazon_reviews.csv", header=True, inferSchema=True)

# View schema and some rows
df.printSchema()
df.show(5)


root
 |-- review_id: string (nullable = true)
 |-- product_title: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- review_body: string (nullable = true)
 |-- star_rating: integer (nullable = true)
 |-- review_date: date (nullable = true)

+--------------------+-----------------+-----------+--------------------+-----------+-----------+
|           review_id|    product_title|customer_id|         review_body|star_rating|review_date|
+--------------------+-----------------+-----------+--------------------+-----------+-----------+
|24d5a403-1a68-4f7...|     Laptop Stand|      71150|North style woman...|          1| 2025-03-08|
|3b258dc9-98aa-490...|  Fitness Tracker|      27958|Whom be side happ...|          2| 2025-02-17|
|648b9d29-994d-43c...|  Fitness Tracker|      49137|Power head fly pr...|          2| 2024-07-21|
|c54560c5-a776-4d9...|  Fitness Tracker|      43921|Reduce only run w...|          4| 2025-02-12|
|cb9ed91e-d288-40e...|Bluetooth Speaker|      6980

## Basic Data Exploration

In [7]:
from pyspark.sql import functions as F

# Count total records
print("Total Records:", df.count())

# Check for missing data in each column
df.select([F.sum(F.col(col).isNull().cast("int")).alias(col) for col in df.columns]).show()

# Summary statistics
df.describe().show()


Total Records: 10000
+---------+-------------+-----------+-----------+-----------+-----------+
|review_id|product_title|customer_id|review_body|star_rating|review_date|
+---------+-------------+-----------+-----------+-----------+-----------+
|        0|            0|          0|          0|          0|          0|
+---------+-------------+-----------+-----------+-----------+-----------+

+-------+--------------------+-----------------+------------------+--------------------+-----------------+
|summary|           review_id|    product_title|       customer_id|         review_body|      star_rating|
+-------+--------------------+-----------------+------------------+--------------------+-----------------+
|  count|               10000|            10000|             10000|               10000|            10000|
|   mean|                NULL|             NULL|        55207.0991|                NULL|           3.0092|
| stddev|                NULL|             NULL|25946.418907249783|      

## Data Cleaning

In [9]:
# Drop rows with null reviews
df_clean = df.dropna(subset=["review_body", "star_rating"])
print(df_clean)

DataFrame[review_id: string, product_title: string, customer_id: int, review_body: string, star_rating: int, review_date: date]


# Analysis

##  Top Reviewed Products

In [12]:
df.groupBy("product_title") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)


+-----------------+-----+
|    product_title|count|
+-----------------+-----+
| Wireless Earbuds| 2029|
|      Smart Watch| 2005|
|Bluetooth Speaker| 1997|
|  Fitness Tracker| 1996|
|     Laptop Stand| 1973|
+-----------------+-----+



## Average Rating Per Product

In [14]:
df.groupBy("product_title") \
  .avg("star_rating") \
  .orderBy("avg(star_rating)", ascending=False) \
  .show(10)


+-----------------+------------------+
|    product_title|  avg(star_rating)|
+-----------------+------------------+
|  Fitness Tracker|  3.05811623246493|
|     Laptop Stand|3.0440952863659403|
|      Smart Watch| 3.000997506234414|
| Wireless Earbuds|2.9990142927550516|
|Bluetooth Speaker| 2.944416624937406|
+-----------------+------------------+



##  Most Active Users

In [16]:
df.groupBy("customer_id") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(10)


+-----------+-----+
|customer_id|count|
+-----------+-----+
|      12436|    3|
|      86231|    3|
|      90777|    3|
|      26478|    3|
|      68828|    3|
|      74826|    3|
|      62401|    3|
|      62534|    3|
|      20317|    3|
|      98665|    3|
+-----------+-----+
only showing top 10 rows



In [17]:
# Derive Insights

**Most reviewed product is “Wireless Earbuds” with 2029 reviews.** 
    
**Product “Fittness Tracker” has the highest average rating of 3.05.**

**Top 5 users wrote over 500 reviews each.**

SyntaxError: invalid character '“' (U+201C) (3765693442.py, line 3)